In [1]:
# !pip install segmentation-models-pytorch
# !pip install torchmetrics
# !pip install mlflow[databricks]

In [2]:
import os

# Replace with your actual workspace URL and token
# os.environ['DATABRICKS_HOST'] = ""
# os.environ['DATABRICKS_TOKEN'] = ""

os.environ['DATABRICKS_HOST'] = ""
os.environ['DATABRICKS_TOKEN'] = ""

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupKFold

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

from torchmetrics.segmentation import DiceScore
from torchmetrics.segmentation import MeanIoU

from tqdm import tqdm
import json

# Configurar el dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def build_color_map_from_json(label_dict):
    color_map_raw = label_dict["mascara_multiclase_color_rgb"]
    return {int(k): tuple(v) for k, v in color_map_raw.items()}

def build_label_map_from_json(label_dict):
    label_map_raw = label_dict["mascara_multiclase_id_png"]
    return {int(k): v for k, v in label_map_raw.items()}

def create_color_overlay(mask, color_map, alpha=0.45):
    h, w = mask.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id, color in color_map.items():
        overlay[mask == class_id] = color

    return overlay

def get_class_centroid(mask, class_id):
    ys, xs = np.where(mask == class_id)
    if len(xs) == 0 or len(ys) == 0:
        return None
    cx = int(np.mean(xs))
    cy = int(np.mean(ys))
    return cx, cy

def show_overlay_with_labels(image, mask, label_dict, alpha=0.45, min_pixels=20):
    label_map = build_label_map_from_json(label_dict)
    color_map = build_color_map_from_json(label_dict)

    # image = cv2.imread(image_path)
    # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

    overlay = create_color_overlay(mask, color_map, alpha=alpha)

    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.imshow(overlay, alpha=alpha)

    unique_vals = np.unique(mask)

    for class_id in unique_vals:
        if class_id == 0:
            continue

        region = (mask == class_id)
        if region.sum() < min_pixels:
            continue

        centroid = get_class_centroid(mask, class_id)
        if centroid is None:
            continue

        x, y = centroid
        class_name = label_map.get(int(class_id), str(class_id))

        plt.text(
            x, y, class_name,
            fontsize=9,
            ha="center",
            va="center",
            color="white",
            bbox=dict(facecolor="black", alpha=0.6, boxstyle="round,pad=0.2")
        )

    plt.title("Overlay con clases anatómicas")
    plt.axis("off")
    plt.show()

In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
# dataset = '/content/drive/MyDrive/Documentos/UniAndes/MAIA/2026-12/Proyecto Despluiegue de Soluciones/Proyecto/Scoliosis_Dataset'
# indice = pd.read_csv(os.path.join(dataset,'indice_dataset.csv'))

In [7]:
dataset = 'Scoliosis_Dataset'
indice = pd.read_csv(os.path.join(dataset,'indice_dataset.csv'))

In [8]:
with open(dataset+os.sep+'diccionario_etiquetas_T1_T12_L1_L5.json', 'r') as f:
    dic_labels = json.load(f)

color_label = dic_labels['mascara_multiclase_color_rgb']

In [9]:
imagenes = pd.DataFrame(columns=['image_path', 'mask_path', 'id_paciente'])
imagenes['image_path'] = dataset +os.sep + indice['grupo'] + os.sep + indice['imagen']
imagenes['mask_path'] = dataset + os.sep + indice['ruta_mascara_multiclase_id_png']
imagenes['id_paciente'] = indice['id_paciente']
# masks_path = masks_path.apple(dataset, indice['grupo'], indice['ruta_mascara_multiclase_id_png'])

In [10]:
imagenes

,image_path,mask_path,id_paciente
0,Scoliosis_Dataset\Normal\N_1.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,1
1,Scoliosis_Dataset\Normal\N_2.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,2
2,Scoliosis_Dataset\Normal\N_3.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,3
3,Scoliosis_Dataset\Normal\N_4.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,4
4,Scoliosis_Dataset\Normal\N_5.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,5
...,...,...,...
245,Scoliosis_Dataset\Scoliosis\S_202.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,202
246,Scoliosis_Dataset\Scoliosis\S_203.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,203
247,Scoliosis_Dataset\Scoliosis\S_204.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,204
248,Scoliosis_Dataset\Scoliosis\S_205.jpg,Scoliosis_Dataset\LabelMultiClass_ID_PNG/Label...,205


In [11]:
MODEL_NAME = 'Segformer'
MODEL_PATH = 'modelos'+os.sep+MODEL_NAME+'.pth'
SEED = 42
NUM_EPOCHS = 100
BATCH_SIZE = 8
PATIENCE = 50
N_SPLITS = 4
LR = 1e-3                   # Learning rate
WD = 1e-4                   # Weight Decay
IMG_SIZE = (256, 256)
IN_CHANNELS = 3
NUM_CLASSES = 18
TRAIN_ENCODER = True
CENTER_IMAGES = True
CLAHE = True
PERSPECTIVE = True

# LOSS = 'CrossEntropyLoss'
LOSS = 'CombinedLoss'
# LOSS = 'DiceLoss'

# ENCODER_NAME = "resnet34"
ENCODER_NAME = "resnet152"
# ENCODER_NAME = "efficientnet-b7"
# ENCODER_NAME = "mit_b5"
# ENCODER_NAME = "tu-swinv2_small_window16_256"
# ENCODER_NAME = 'tu-sam2_hiera_large'

ENCODER_WEIGHTS = "imagenet"
# ENCODER_WEIGHTS = None

# DECODER_ATTENTION = 'scse'
DECODER_ATTENTION = None

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

if IN_CHANNELS == 1:
  MEAN = 0.44531356896770125
  STD = 0.2692461874154524

# CE_WEIGHT = 0.5
# DICE_WEIGHT = 0.5
# BASE_CHANNELS = 32

In [12]:
import mlflow

# Set the tracking URI to the Databricks workspace
mlflow.set_tracking_uri("databricks")

# Set the experiment name (must start with /Users/<email>/ or /Shared/)
experiment = mlflow.set_experiment("/"+MODEL_NAME)

In [13]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        # logits: [N, C, H, W], targets: [N, H, W]
        num_classes = logits.shape[1]
        probs = torch.nn.functional.softmax(logits, dim=1)

        # Convert targets to one-hot: [N, C, H, W]
        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()

        # Calculate intersection and cardinality
        dims = (0, 2, 3) # Sum over batch, height, and width
        intersection = torch.sum(probs * targets_one_hot, dim=dims)
        union = torch.sum(probs + targets_one_hot, dim=dims)

        dice_score = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice_score.mean() # Return mean loss across classes

class CombinedLoss(nn.Module):
    def __init__(self, ce_weight=1, dice_weight=1):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.ce_weight * ce_loss + self.dice_weight * dice_loss

In [14]:
# Obtencion Rutas de imagenes

image_files = imagenes['image_path'].to_numpy()
mask_files = imagenes['mask_path'].to_numpy()

In [15]:
# Split: 80% entrenamiento, 20% validacion
# train_img, val_img, train_mask, val_mask = train_test_split(
#     image_files, mask_files, test_size=0.2, random_state=42
# )

gkf = GroupKFold(n_splits=N_SPLITS)

In [16]:
groups = imagenes['id_paciente']
splits = gkf.split(image_files, mask_files, groups=groups)
for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    train_df = imagenes.iloc[train_idx].reset_index(drop=True)
    val_df = imagenes.iloc[val_idx].reset_index(drop=True)

    print(f"Fold {fold}")
    print("Train samples:", len(train_df))
    print("Val samples:", len(val_df))
    print("Train patients:", train_df["id_paciente"].nunique())
    print("Val patients:", val_df["id_paciente"].nunique())
    print("-" * 40)

Fold 1
Train samples: 187
Val samples: 63
Train patients: 149
Val patients: 50
----------------------------------------
Fold 2
Train samples: 187
Val samples: 63
Train patients: 149
Val patients: 50
----------------------------------------
Fold 3
Train samples: 188
Val samples: 62
Train patients: 150
Val patients: 49
----------------------------------------
Fold 4
Train samples: 188
Val samples: 62
Train patients: 149
Val patients: 50
----------------------------------------


In [17]:
def preproc_img(img, size: tuple) -> np.ndarray:
    """
    Pipeline de preprocesamiento de radiografía:
      1. Lectura en escala de grises
      2. CLAHE (mejora contraste local)
      3. Letterbox resize (mantiene aspecto + padding negro)
      4. Normalización a float32 en [0, 1]
    """

    # CLAHE: mejora el contraste local sin saturar zonas uniformes
    # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    # img = clahe.apply(img)

    # Letterbox: escalar con el factor que no supere ninguna dimensión
    # th, tw = size
    # h, w, c   = img.shape
    # s      = min(tw / w, th / h)
    # nw, nh = int(w * s), int(h * s)
    # img_r  = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LANCZOS4)
    #
    # # Centrar la imagen escalada en un canvas de padding negro
    # canvas = np.zeros((th, tw, c), np.uint8)
    # yo, xo = (th - nh) // 2, (tw - nw) // 2
    # canvas[yo:yo+nh, xo:xo+nw] = img_r
    #
    # return canvas
    #get size
    if IN_CHANNELS == 1:
      return(preproc_mask(img, size))
    else:
      height, width, channels = img.shape

    # Create a black image
    x = height if height > width else width
    y = height if height > width else width
    square= np.zeros((x,y,channels), np.uint8)

    #
    #This does the job
    #
    square[int((y-height)/2):int(y-(y-height)/2), int((x-width)/2):int(x-(x-width)/2)] = img

    return square


def preproc_mask(mask, size: tuple) -> np.ndarray:
    """
    Redimensiona la mascara uint16 con interpolacion NEAREST.
    NEAREST es obligatorio: preserva los IDs exactos de vertebras.
    PIL maneja uint16 correctamente; cv2 lo truncaria a uint8.
    """
    # th, tw = size
    # h, w   = mask.shape
    # s      = min(tw / w, th / h)
    # nw, nh = int(w * s), int(h * s)
    # m_r    = cv2.resize(mask, (nw, nh), cv2.INTER_NEAREST)
    #
    # canvas = np.zeros((th, tw), np.uint16)
    # yo, xo = (th - nh) // 2, (tw - nw) // 2
    # canvas[yo:yo+nh, xo:xo+nw] = m_r
    # return canvas.astype(np.int64)
    height, width = mask.shape
    # Create a black image
    x = height if height > width else width
    y = height if height > width else width
    square= np.zeros((x,y), np.int64)
    #
    #This does the job
    #
    square[int((y-height)/2):int(y-(y-height)/2), int((x-width)/2):int(x-(x-width)/2)] = mask

    return square.astype('uint8')

In [18]:
class CustomSegmentationDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row['image_path']
        mask_path = row['mask_path']
        try:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)
            # if IN_CHANNELS == 1:
            #   image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            #   if CLAHE:
            #     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            #     image = clahe.apply(image)
            #
            # # Convertir a enteros para CrossEntropyLoss
            # # mask = mask.astype(np.int64)
            # else:
            #   image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)
            #   if CLAHE:
            #       image = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            #       clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            #       image[0] = clahe.apply(image[0])
            #       image = cv2.cvtColor(image, cv2.COLOR_LAB2RGB)
            #
            # if CENTER_IMAGES:
            #     image = preproc_img(image, IMG_SIZE)
            #     mask = preproc_mask(mask, IMG_SIZE)

            if self.transform:
                augmented = self.transform(image=image, mask=mask)
                image = augmented['image']
                mask = augmented['mask']

        except:
            # print(image_path, mask_path)
            new_idx = (idx + 1)
            return self.__getitem__(new_idx)



        # ELIMINAR: mask = mask.unsqueeze(0)
        # Para multiclase, la máscara debe quedar con dimensión [H, W]

        return image, mask

In [19]:
# test_img = cv2.imread(imagenes['image_path'].to_numpy()[0], cv2.COLOR_BGR2RGB)
# test_img = cv2.imread(imagenes['image_path'].to_numpy()[0], cv2.IMREAD_GRAYSCALE)
# test_img = preproc_img(test_img, IMG_SIZE)
# plt.imshow(test_img)

# test_mask = cv2.imread(imagenes['mask_path'].to_numpy()[0], cv2.IMREAD_GRAYSCALE)
# test_mask = preproc_mask(test_mask, IMG_SIZE)
# plt.imshow(test_mask)

In [20]:
# --- TRANSFORMACIONES (Albumentations) ---
normalization = A.Normalize(mean=MEAN, std=STD)
clahe = A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0)

# Transformaciones de entrenamiento: redimensionamiento, aumento de datos y normalización
train_transforms = []
if CENTER_IMAGES:
    train_transforms+=[A.LongestMaxSize(IMG_SIZE[0]),
    A.PadIfNeeded(IMG_SIZE[0], IMG_SIZE[1])]
else:
    train_transforms.append(A.Resize(IMG_SIZE[0],IMG_SIZE[1]))

train_transforms+=[
    A.HorizontalFlip(p=0.2),
    A.Rotate(limit=(-10, 10), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=(-0.1,0.1), p=0.5)
]

if PERSPECTIVE:
    train_transforms.append(A.Perspective(scale=(0.01, 0.05)))
if CLAHE:
    train_transforms.append(clahe)

# train_transforms.append(normalization)
train_transforms.append(ToTensorV2())

train_transform = A.Compose(train_transforms)
# train_transform = A.Compose([
#     A.LongestMaxSize(IMG_SIZE[0]),
#     A.PadIfNeeded(IMG_SIZE[0], IMG_SIZE[1]),
#     A.HorizontalFlip(p=0.2),
#     A.Rotate(limit=(-10, 10), p=1.0),
#     # A.VerticalFlip(p=0.5),
#     A.RandomBrightnessContrast(brightness_limit=(-0.1,0.1), p=0.5),
#     A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
#     # A.Normalize(mean=MEAN, std=STD),
#     # A.Resize(IMG_SIZE[0],IMG_SIZE[1]),
#     ToTensorV2(),
# ])

# Transformaciones de validación/prueba: solo redimensionamiento y normalización
val_transforms = []
if CENTER_IMAGES:
    val_transforms+=[A.LongestMaxSize(IMG_SIZE[0]),
    A.PadIfNeeded(IMG_SIZE[0], IMG_SIZE[1])]
else:
    val_transforms.append(A.Resize(IMG_SIZE[0],IMG_SIZE[1]))

if CLAHE:
    val_transforms.append(clahe)

# val_transforms.append(normalization)
val_transforms.append(ToTensorV2())

val_transform = A.Compose(val_transforms)

# val_transform = A.Compose([
#     A.LongestMaxSize(IMG_SIZE[0]),
#     A.PadIfNeeded(IMG_SIZE[0], IMG_SIZE[1]),
#     A.Normalize(mean=MEAN, std=STD),
#     A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
#     # A.Resize(IMG_SIZE[0],IMG_SIZE[1]),
#     ToTensorV2(),
# ])

# --- DATALOADERS ---
# train_dataset = CustomSegmentationDataset(train_img, train_mask, transform=train_transform)
# val_dataset = CustomSegmentationDataset(val_img, val_mask, transform=val_transform)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# print(f"Imágenes de entrenamiento: {len(train_dataset)}")
# print(f"Imágenes de validación: {len(val_dataset)}")

In [21]:
# show_overlay_with_labels(
#     image_files[0],
#     mask_files[0],
#     dic_labels
# )

In [22]:
# transformed = train_transform(image=cv2.imread(image_files[150], cv2.COLOR_BGR2RGB), mask=cv2.imread(mask_files[150], cv2.IMREAD_GRAYSCALE))
# print(cv2.imread(image_files[150], cv2.COLOR_BGR2RGB).shape)
# show_overlay_with_labels(
#     transformed['image'],
#     transformed['mask'],
#     dic_labels
# )

In [23]:
# image, mask = val_dataset[19]
# print(image.shape)
# image = image.permute(1, 2, 0).cpu().numpy()
# mask = mask.squeeze().cpu().numpy()
# show_overlay_with_labels(image, mask, dic_labels)

In [24]:
# train_transform(image=test_img, mask=test_mask)

In [25]:
# train_dataset.__getitem__(10)[0]

In [26]:
# imagen=cv2.imread(imagenes['image_path'][10], cv2.COLOR_BGR2RGB)
# mask=cv2.imread(imagenes['mask_path'][10], cv2.IMREAD_GRAYSCALE)

In [27]:
# train_transform(image=imagen, mask=mask)

In [28]:
# train_dataset.__getitem__(123)

In [29]:
#to view every iterated batch
# for batch_images, batch_labels in train_loader:
    # print(f"Batch shape: {batch_images.shape}, Labels: {batch_labels.shape}")

In [30]:
# class DiceLoss(nn.Module):
#     def __init__(self, num_classes = NUM_CLASSES):
#         super(DiceLoss, self).__init__()
#         self.dice_score = DiceScore(num_classes=num_classes, average='macro', input_format='index')
#
#     def forward(self, logits, masks):
#         preds = torch.argmax(logits, dim=1)
#         print(preds)
#         return 1.0 - self.dice_score(preds, masks)

In [ ]:
dice_score = DiceScore(num_classes=NUM_CLASSES, average="macro", input_format='index')
IoU = MeanIoU(num_classes=NUM_CLASSES, input_format='index').to(device)
splits = gkf.split(image_files, mask_files, groups=groups)

fold_dice = []
fold_train_losses = []
fold_val_losses = []
fold_val_dice = []
best_fold = 1
best_val_dice = 0
for fold, (train_idx, val_idx) in tqdm(enumerate(splits)):
    print(f'Entrenando Fold: {fold+1}')
    train_df = imagenes.iloc[train_idx].reset_index(drop=True)
    val_df = imagenes.iloc[val_idx].reset_index(drop=True)

    # --- DATALOADERS ---
    train_dataset = CustomSegmentationDataset(train_df, transform=train_transform)
    val_dataset = CustomSegmentationDataset(val_df, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # image, mask = train_dataset[19]
    # print(image.shape)
    # image = image.permute(1, 2, 0).cpu().numpy()
    # mask = mask.squeeze().cpu().numpy()
    # show_overlay_with_labels(image, mask, dic_labels)

    if MODEL_NAME == 'U-net++':
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=IN_CHANNELS,
            classes=NUM_CLASSES,
            decoder_attention_type = DECODER_ATTENTION,
            # pooling='max',
            # dropout = 0.5
        )
    elif MODEL_NAME == 'Segformer':
        model = smp.Segformer(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=IN_CHANNELS,
            classes=NUM_CLASSES,
            # aux_params={'classes':NUM_CLASSES, 'dropout':0.2}
        )
    if MODEL_NAME == 'U-net':
        model = smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=IN_CHANNELS,
            classes=NUM_CLASSES,
            decoder_attention_type = DECODER_ATTENTION,
            # pooling='max',
            # dropout = 0.5
        )

    model = model.to(device)

    # Definir la función de pérdida y el optimizador
    # Usamos BCEWithLogitsLoss porque el modelo no tiene una capa sigmoide final por defecto
    # criterion = nn.BCEWithLogitsLoss()

    if LOSS == 'CrossEntropyLoss':
        criterion = nn.CrossEntropyLoss()

    elif LOSS == 'DiceLoss':
        criterion =  DiceLoss()

    elif LOSS == 'CombinedLoss':
        criterion = CombinedLoss()

    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

    if not TRAIN_ENCODER:
        for param in model.encoder.parameters():
            param.requires_grad = False


    # Definir el scheduler
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',    # 'min' porque quieres que la pérdida disminuya
        factor=0.1,    # Reduce el LR al 10% (multiplica por 0.1)
        patience=PATIENCE,    # Espera 5 épocas sin mejora antes de reducir
        threshold=1e-4, # Cambio mínimo para considerar que hay mejora
    )

    train_losses = np.zeros(NUM_EPOCHS)
    val_losses = np.zeros(NUM_EPOCHS)
    val_dice = np.zeros(NUM_EPOCHS)
    best_epoch = 0
    best_metric = 0
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0.0

        for images, masks in train_loader:
            images, masks = images.float().to(device), masks.long().to(device) # <--- Asegurar .long()

            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)

            # pred_mask = torch.argmax(outputs, dim=1)
            loss = criterion(outputs, masks)
            # loss = criterion(pred_mask, masks)

            # Backward pass y optimización
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validación simple
        model.eval()
        val_loss = 0.0
        dice_score_val = 0.0
        IoU_val = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.float().to(device), masks.long().to(device)
                outputs = model(images)
                # pred_mask = torch.argmax(outputs, dim=1)
                loss = criterion(outputs, masks)
                # loss = criterion(pred_mask, masks)
                val_loss += loss.item()
                pred_mask = torch.argmax(outputs, dim=1)
                dice_score_val += dice_score(pred_mask.cpu(), masks.to(torch.long).cpu())
                IoU_val += IoU(pred_mask.cpu(), masks.to(torch.long).cpu())

        if dice_score_val/len(val_loader) > best_metric:
            best_metric = dice_score_val/len(val_loader)
            torch.save(model.state_dict(), 'modelos'+os.sep+MODEL_NAME+f'fold_{fold+1}'+'.pth')
            best_epoch = epoch + 1

        train_losses[epoch] = train_loss / len(train_loader)
        val_losses[epoch] = val_loss / len(val_loader)

        scheduler.step(val_loss/len(val_loader))
        val_dice[epoch] = dice_score_val/len(val_loader)
        print(f"Fold {fold+1} Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val DICE Score: {dice_score_val/len(val_loader):.4f} | Val IoU Score: {IoU_val/len(val_loader):.4f}")

    print('Best Epoch: ', best_epoch, ' Best metric: ', best_metric)

    if best_metric >= best_val_dice:
        best_val_dice = best_metric
        best_fold = fold + 1
        torch.save(model.state_dict(), MODEL_PATH)

    fold_dice.append(best_metric)
    fold_train_losses.append(train_losses)
    fold_val_losses.append(val_losses)
    fold_val_dice.append(val_dice)

mean_val_dice = sum(fold_dice) / len(fold_dice)
print(f'Val DICE Score promedio: {mean_val_dice:.4f}')
print(f'Mejor Fold: {best_fold} | Val Dice Score: {best_val_dice}')

0it [00:00, ?it/s]

Entrenando Fold: 1
Fold 1 Epoch 1/100 | Train Loss: 1.7160 | Val Loss: 16.5087 | Val DICE Score: 0.0767 | Val IoU Score: 0.0697
Fold 1 Epoch 2/100 | Train Loss: 1.0930 | Val Loss: 1.0999 | Val DICE Score: 0.1435 | Val IoU Score: 0.1121
Fold 1 Epoch 3/100 | Train Loss: 1.0222 | Val Loss: 1.0070 | Val DICE Score: 0.1590 | Val IoU Score: 0.1241
Fold 1 Epoch 4/100 | Train Loss: 0.9730 | Val Loss: 1.0417 | Val DICE Score: 0.1496 | Val IoU Score: 0.1175
Fold 1 Epoch 5/100 | Train Loss: 0.9695 | Val Loss: 0.9975 | Val DICE Score: 0.1828 | Val IoU Score: 0.1425
Fold 1 Epoch 6/100 | Train Loss: 0.9444 | Val Loss: 0.9744 | Val DICE Score: 0.2065 | Val IoU Score: 0.1616
Fold 1 Epoch 7/100 | Train Loss: 0.9246 | Val Loss: 0.9667 | Val DICE Score: 0.2265 | Val IoU Score: 0.1742
Fold 1 Epoch 8/100 | Train Loss: 0.9099 | Val Loss: 0.9340 | Val DICE Score: 0.2565 | Val IoU Score: 0.1986
Fold 1 Epoch 9/100 | Train Loss: 0.8782 | Val Loss: 0.9539 | Val DICE Score: 0.2260 | Val IoU Score: 0.1773
Fold 1 E

1it [10:11, 611.96s/it]

Entrenando Fold: 2
Fold 2 Epoch 1/100 | Train Loss: 1.7781 | Val Loss: 2.6295 | Val DICE Score: 0.0769 | Val IoU Score: 0.0678
Fold 2 Epoch 2/100 | Train Loss: 1.0824 | Val Loss: 1.4097 | Val DICE Score: 0.0771 | Val IoU Score: 0.0686
Fold 2 Epoch 3/100 | Train Loss: 1.0146 | Val Loss: 1.0693 | Val DICE Score: 0.0929 | Val IoU Score: 0.0788
Fold 2 Epoch 4/100 | Train Loss: 0.9739 | Val Loss: 0.9905 | Val DICE Score: 0.2078 | Val IoU Score: 0.1627
Fold 2 Epoch 5/100 | Train Loss: 0.9573 | Val Loss: 0.9651 | Val DICE Score: 0.2182 | Val IoU Score: 0.1676
Fold 2 Epoch 6/100 | Train Loss: 0.9184 | Val Loss: 0.9759 | Val DICE Score: 0.2201 | Val IoU Score: 0.1689
Fold 2 Epoch 7/100 | Train Loss: 0.9378 | Val Loss: 0.9368 | Val DICE Score: 0.2420 | Val IoU Score: 0.1877
Fold 2 Epoch 8/100 | Train Loss: 0.9002 | Val Loss: 0.9217 | Val DICE Score: 0.2498 | Val IoU Score: 0.1909
Fold 2 Epoch 9/100 | Train Loss: 0.8609 | Val Loss: 0.9655 | Val DICE Score: 0.2173 | Val IoU Score: 0.1685
Fold 2 Ep

2it [20:02, 599.11s/it]

Entrenando Fold: 3
Fold 3 Epoch 1/100 | Train Loss: 1.8556 | Val Loss: 1.5813 | Val DICE Score: 0.0752 | Val IoU Score: 0.0675
Fold 3 Epoch 2/100 | Train Loss: 1.1137 | Val Loss: 1.2214 | Val DICE Score: 0.0857 | Val IoU Score: 0.0747
Fold 3 Epoch 3/100 | Train Loss: 1.0610 | Val Loss: 1.0196 | Val DICE Score: 0.1273 | Val IoU Score: 0.1032
Fold 3 Epoch 4/100 | Train Loss: 1.0132 | Val Loss: 1.0130 | Val DICE Score: 0.1613 | Val IoU Score: 0.1246
Fold 3 Epoch 5/100 | Train Loss: 0.9659 | Val Loss: 0.9498 | Val DICE Score: 0.2127 | Val IoU Score: 0.1644
Fold 3 Epoch 6/100 | Train Loss: 0.9292 | Val Loss: 0.9976 | Val DICE Score: 0.1716 | Val IoU Score: 0.1349
Fold 3 Epoch 7/100 | Train Loss: 0.9451 | Val Loss: 0.9346 | Val DICE Score: 0.2249 | Val IoU Score: 0.1772
Fold 3 Epoch 8/100 | Train Loss: 0.9035 | Val Loss: 0.9301 | Val DICE Score: 0.2283 | Val IoU Score: 0.1809
Fold 3 Epoch 9/100 | Train Loss: 0.8941 | Val Loss: 0.9467 | Val DICE Score: 0.2172 | Val IoU Score: 0.1707
Fold 3 Ep

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))

In [ ]:
with mlflow.start_run(experiment_id=experiment.experiment_id):

    mlflow.log_params({'best_fold': best_fold, 'EPOCHS': NUM_EPOCHS, 'SEED': SEED, 'BATCH_SIZE': BATCH_SIZE, 'PATIENCE': PATIENCE, 'N_SPLITS': N_SPLITS, 'LR': LR, 'WD': WD, 'IMG_SIZE': IMG_SIZE, 'IN_CHANNELS': IN_CHANNELS, 'TRAIN_ENCODER': TRAIN_ENCODER, 'CENTER_IMAGES': CENTER_IMAGES, 'CLAHE': CLAHE, 'PERSPECTIVE': PERSPECTIVE ,'LOSS': LOSS, 'ENCODER_NAME': ENCODER_NAME, 'DECODER_ATTENTION': DECODER_ATTENTION})
    mlflow.log_metric(f"val_mean_dice_score", mean_val_dice)
    mlflow.log_metric(f"best_val_dice_score", best_val_dice)
    mlflow.pytorch.log_model(model, name=MODEL_NAME)

    for step, value in enumerate(fold_train_losses[best_fold-1]):
        mlflow.log_metric('train_hist', value, step=step)
    for step, value in enumerate(fold_val_losses[best_fold-1]):
        mlflow.log_metric('val_hist', value, step=step)

    # Log best parameters and score
    # mlflow.log_params(grid.best_params_)
    # mlflow.log_metric("best_score", grid.best_score_)

    # # Log the best model
    # mlflow.sklearn.log_model(grid.best_estimator_, "best_model")

In [ ]:
plt.figure()
i=0
for train_losses, val_losses in zip(fold_train_losses, fold_val_losses):
    plt.plot(train_losses, label=f"Fold {i+1} - Train Loss")
    plt.plot(val_losses, label=f"Fold {i+1} Val Loss")
    i+=1
plt.xlabel('Epochs')
plt.title("Loss durante el entrenamiento de cada fold")
plt.legend()

In [ ]:
plt.figure()
i=0
for i, dice in enumerate(fold_val_dice):
    plt.plot(dice, label=f"Fold {i+1}")
    i+=1
plt.xlabel('Epochs')
plt.ylabel("Validation Dice")
plt.title("Evolución del Dice Score por Fold")
plt.legend()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

def build_color_map_from_json(label_dict):
    color_map_raw = label_dict["mascara_multiclase_color_rgb"]
    return {int(k): tuple(v) for k, v in color_map_raw.items()}

def build_label_map_from_json(label_dict):
    label_map_raw = label_dict["mascara_multiclase_id_png"]
    return {int(k): v for k, v in label_map_raw.items()}

def create_color_overlay(mask, color_map):
    h, w = mask.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id, color in color_map.items():
        overlay[mask == class_id] = color

    return overlay

def get_class_centroid(mask, class_id):
    ys, xs = np.where(mask == class_id)
    if len(xs) == 0 or len(ys) == 0:
        return None
    cx = int(np.mean(xs))
    cy = int(np.mean(ys))
    return cx, cy

def draw_labels_on_axis(ax, mask, label_map, min_pixels=20):
    unique_vals = np.unique(mask)

    for class_id in unique_vals:
        if class_id == 0:
            continue

        region = (mask == class_id)
        if region.sum() < min_pixels:
            continue

        centroid = get_class_centroid(mask, class_id)
        if centroid is None:
            continue

        x, y = centroid
        class_name = label_map.get(int(class_id), str(class_id))

        ax.text(
            x, y, class_name,
            fontsize=9,
            ha="center",
            va="center",
            color="white",
            bbox=dict(facecolor="black", alpha=0.6, boxstyle="round,pad=0.2")
        )

def show_gt_vs_pred_with_labels(
    model,
    dataset,
    device,
    idx,
    label_dict,
    alpha=0.45,
    min_pixels=20
):
    model.eval()

    label_map = build_label_map_from_json(label_dict)
    color_map = build_color_map_from_json(label_dict)

    sample = dataset[idx]
    if sample is None:
        print(f"No se pudo cargar la muestra {idx}")
        return

    image_tensor, gt_mask = sample
    image_batch = image_tensor.float().unsqueeze(0).to(device)

    with torch.no_grad():
        if device.type == "cuda":
            with torch.amp.autocast(device_type="cuda"):
                logits = model(image_batch)
        else:
            logits = model(image_batch)

    pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu()

    dice_score = DiceScore(num_classes=NUM_CLASSES, average="none", input_format='index')
    sample_dice = dice_score(pred_mask, gt_mask.to(torch.long).cpu())

    pred_mask = pred_mask.numpy()
    gt_mask = gt_mask.cpu().numpy()

    image = image_tensor.cpu().numpy()
    if image.shape[0] == 1:
        image = image.squeeze(0)
        image_cmap = "gray"
    else:
        image = np.transpose(image, (1, 2, 0))
        image_cmap = None

    # image = np.clip(image, 0, 1)

    gt_overlay = create_color_overlay(gt_mask, color_map)
    pred_overlay = create_color_overlay(pred_mask, color_map)

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    axes[0].imshow(image, cmap=image_cmap)
    axes[0].set_title("Imagen")
    axes[0].axis("off")

    axes[1].imshow(image, cmap=image_cmap)
    axes[1].imshow(gt_overlay, alpha=alpha)
    draw_labels_on_axis(axes[1], gt_mask, label_map, min_pixels=min_pixels)
    axes[1].set_title("Ground Truth con etiquetas")
    axes[1].axis("off")

    axes[2].imshow(image, cmap=image_cmap)
    axes[2].imshow(pred_overlay, alpha=alpha)
    draw_labels_on_axis(axes[2], pred_mask, label_map, min_pixels=min_pixels)
    axes[2].set_title("Predicción con etiquetas")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    gt_classes = np.unique(gt_mask)
    pred_classes = np.unique(pred_mask)
    print('Dice Score por clase: ')
    for i,j in zip(dic_labels['mascara_multiclase_id_png'], sample_dice):
        print(dic_labels['mascara_multiclase_id_png'][i],j.numpy())

    # print(f"DICE Score: {sample_dice:.4f}")

    print("\nClases presentes en Ground Truth:")
    for c in gt_classes:
        name = label_map.get(int(c), f"Clase {c}")
        print(f"  {c} → {name}")

    print("\nClases presentes en Predicción:")
    for c in pred_classes:
        name = label_map.get(int(c), f"Clase {c}")
        print(f"  {c} → {name}")

In [ ]:
dataset=CustomSegmentationDataset(imagenes, transform=train_transform)

In [ ]:
show_gt_vs_pred_with_labels(
    model=model,
    dataset=dataset,
    device=device,
    idx=0,
    label_dict=dic_labels,
    alpha=0.45,
    min_pixels=20
)

In [ ]:
x = range(len(dice_score_val.numpy()))
f, ax = plt.subplots()
ax.plot(x, dice_score_val.numpy())
ax.set_xticks(x)
labels = [dic_labels['mascara_multiclase_id_png'][i] for i  in dic_labels['mascara_multiclase_id_png']]
ax.set_xticklabels(labels)
ax.figure.set_size_inches(15, 5)
ax.set_xlabel('Label')
ax.set_ylabel('mean DICE score')

In [ ]:
dice_score = DiceScore(num_classes=NUM_CLASSES, average="macro", input_format='index')
IoU = MeanIoU(num_classes=NUM_CLASSES, input_format='index').to(device)

mean_dice_score = 0
mean_IoU = 0
with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.long().to(device)
            outputs = model(images)
            # pred_mask = torch.argmax(outputs, dim=1)
            loss = criterion(outputs, masks)
            # loss = criterion(pred_mask, masks)
            pred_mask = torch.argmax(outputs, dim=1)
            mean_dice_score += dice_score(pred_mask.cpu(), masks.to(torch.long).cpu())
            mean_IoU += IoU(pred_mask.cpu(), masks.to(torch.long).cpu())

mean_dice_score = mean_dice_score/len(val_loader)
print('Dice Score: ', mean_dice_score)

mean_IoU = mean_IoU/len(val_loader)
print('IoU: ', mean_IoU)

In [ ]:
type(np.array(color_label['0']))

In [ ]:
torch.save(model.state_dict(), 'modelo_draft_3_U++.pth')

In [ ]:
with mlflow.start_run():
    mlflow.log_param("env", "google_colab")
    mlflow.log_metric("status", 1.0)
    print("Run logged to Databricks!")

In [ ]:
model_loaded = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=18,
)
model_loaded.load_state_dict(torch.load('modelo_draft.pth', weights_only=True))

In [ ]:

# Seleccionar un índice aleatorio del dataset de validación
idx = random.randint(0, len(val_dataset) - 1)

# Obtener imagen y máscara real (la imagen ya está transformada y normalizada)
image_tensor, mask_tensor = val_dataset[idx]

# Añadir dimensión de batch y mover a dispositivo
image_input = image_tensor.unsqueeze(0)

# Inferencia
# Inferencia
model_loaded.eval()
with torch.no_grad():
    logits = model_loaded(image_input) # Salida: [1, 15, H, W]

    # En lugar de usar sigmoide y umbral (> 0.5), buscamos el índice con mayor valor (argmax)
    pred_mask = torch.argmax(logits, dim=1) # Salida: [1, H, W] con valores de 0 a 14

# --- PREPARACIÓN PARA VISUALIZACIÓN ---
# Deshacer la normalización para visualizar la imagen a color correctamente
inv_normalize = A.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225],
    max_pixel_value=1.0
)

# Convertir tensores a arrays de numpy
image_vis = image_tensor.permute(1, 2, 0).cpu().numpy()
image_vis = (image_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)

# Convertir a numpy para visualización
mask_vis = mask_tensor.squeeze().cpu().numpy()
pred_vis = pred_mask.squeeze().cpu().numpy()

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_vis)
axes[0].set_title("Imagen Original")
axes[0].axis('off')

axes[1].imshow(mask_vis, cmap='gray')
axes[1].set_title("Máscara Real (Ground Truth)")
axes[1].axis('off')

axes[2].imshow(pred_vis, cmap='gray')
axes[2].set_title("Predicción (U-Net)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

dice_score = DiceScore(num_classes=18, average="macro")
print(dice_score(pred_mask, mask_tensor.unsqueeze(0)))